# 04_Feature_Engineering

This notebook consolidates all features discovered and built during
`03_Exploratory_Data_Analysis.ipynb` into two clean, documented, analysis-ready tables:

- **`items_master`** — item-level grain, one row per `order_item`. Used for
  product/seller-focused analysis (category, weight, distance, freight).
- **`orders_master`** — order-level grain, one row per `order_id`. Used for
  KPI reporting and dashboarding (on-time rate, satisfaction, revenue).

Nothing new is introduced here; this is a consolidation and documentation step.

## 0. Imports & Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import unicodedata

pd.set_option('display.max_columns', None)

In [2]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

order_items_date_columns = ['shipping_limit_date']
review_date_columns = ['review_creation_date', 'review_answer_timestamp']

orders_cleaned      = pd.read_csv('../data/interim/olist_orders_cleaned.csv', parse_dates=date_cols)
order_items_cleaned = pd.read_csv('../data/interim/olist_order_items_cleaned.csv', parse_dates=order_items_date_columns)
customers_cleaned   = pd.read_csv('../data/interim/olist_customers_cleaned.csv')
payments_cleaned    = pd.read_csv('../data/interim/olist_ payments_cleaned.csv')
products_cleaned    = pd.read_csv('../data/interim/olist_products_cleaned.csv')
translation_cleaned = pd.read_csv('../data/interim/olist_ translation_cleaned.csv')
geolocation_cleaned = pd.read_csv('../data/interim/olist_geolocation_cleaned.csv')
reviews_cleaned     = pd.read_csv('../data/interim/olist_ reviews_cleaned.csv', parse_dates=review_date_columns)
sellers_cleaned     = pd.read_csv('../data/interim/olist_sellers_cleaned.csv')

**Note:** `orders_cleaned` as loaded from `02_Data_Preparation` already contains the
duration columns (`purchase_to_approved_hours`, `approval_to_carrier_hours`,
`carrier_to_customer_hours`, `purchase_to_customer_hours`) computed during that phase.
They are re-derived explicitly below only for clarity and to keep this notebook
self-contained and independently re-runnable.

## 1. Category Translation Fix (prerequisite for `items_master`)

Two product categories (`portateis_cozinha_e_preparadores_de_alimentos`, `pc_gamer`)
are missing from the original translation table. This was identified and resolved in
EDA (cells 51–58) — reproduced here since `items_master` depends on it.

In [3]:
missing_rows = pd.DataFrame({
    'product_category_name': [
        'portateis_cozinha_e_preparadores_de_alimentos',
        'pc_gamer'
    ],
    'product_category_name_english': [
        'kitchen_portables_and_food_preparators',
        'pc_gamer'
    ]
})
translation_cleaned = pd.concat([translation_cleaned, missing_rows], ignore_index=True)
print(f"translation_cleaned rows after fix: {len(translation_cleaned)}")  # 71 + 2 = 73

products_cleaned = products_cleaned.merge(
    translation_cleaned[['product_category_name', 'product_category_name_english']],
    on='product_category_name',
    how='left'
)
print(f"products_cleaned rows after merge: {len(products_cleaned)}")  # must stay 32951

translation_cleaned rows after fix: 73
products_cleaned rows after merge: 32951


## 2. Orders-level Features (`orders_fe`)

Rebuilds the lifecycle duration features, the `has_invalid_duration` data-quality flag,
and the delivery-performance features (`is_late`, `delay_vs_estimate_days`) exactly as
validated in EDA cells 15, 17–18, and 101–102.

In [ ]:
orders_fe = orders_cleaned.copy()

# Lifecycle duration features (hours) 
orders_fe['purchase_to_approved_hours'] = (
    orders_fe['order_approved_at'] - orders_fe['order_purchase_timestamp']
).dt.total_seconds() / 3600

orders_fe['approval_to_carrier_hours'] = (
    orders_fe['order_delivered_carrier_date'] - orders_fe['order_approved_at']
).dt.total_seconds() / 3600

orders_fe['carrier_to_customer_hours'] = (
    orders_fe['order_delivered_customer_date'] - orders_fe['order_delivered_carrier_date']
).dt.total_seconds() / 3600

orders_fe['purchase_to_customer_hours'] = (
    orders_fe['order_delivered_customer_date'] - orders_fe['order_purchase_timestamp']
).dt.total_seconds() / 3600

# Data quality flag
orders_fe['has_invalid_duration'] = (
    (orders_fe['approval_to_carrier_hours'] < 0) |
    (orders_fe['carrier_to_customer_hours'] < 0)
)

# Delivery performance features
orders_fe['delay_vs_estimate_days'] = (
    orders_fe['order_delivered_customer_date'] - orders_fe['order_estimated_delivery_date']
).dt.days
orders_fe['is_late'] = orders_fe['delay_vs_estimate_days'] > 0

# On-time flag as defined in EDA 
orders_fe['is_on_time'] = (
    orders_fe['order_delivered_customer_date'] <= orders_fe['order_estimated_delivery_date']
)

print(f"orders_fe shape: {orders_fe.shape}")
print(f"has_invalid_duration count: {orders_fe['has_invalid_duration'].sum()}")

orders_fe shape: (99441, 17)
has_invalid_duration count: 1382


**Note on `is_late` vs `is_on_time`:** these are not simple opposites. `is_late` is
`NaN`-safe False for undelivered orders (no delivery date to compare), while
`is_on_time` follows the exact `<=` comparison used in the original on-time-rate KPI
(EDA cell 17). Both are kept because they served two different validated analyses;
downstream users should filter on `order_delivered_customer_date.notna()` before
using either for a KPI, exactly as done in EDA.

## 3. Reviews — Deduplicated (`reviews_fe`)

551 orders had more than one review row; 202 of those had genuinely different scores
across duplicates. Rule adopted in EDA (cell 99): **keep the most recent review per
order**, based on `review_creation_date`.

In [6]:
reviews_fe = (
    reviews_cleaned
    .sort_values('review_creation_date')
    .drop_duplicates('order_id', keep='last')
)[['order_id', 'review_score']]

print(f"reviews_fe rows: {len(reviews_fe)}")
print(f"should equal unique orders in reviews_cleaned: {reviews_cleaned['order_id'].nunique()}")

reviews_fe rows: 98673
should equal unique orders in reviews_cleaned: 98673


## 4. Geolocation — Brazil Bounds Filter + ZIP-level Aggregation

Coordinates outside Brazil's real geographic bounds (33 rows, e.g. matching Portugal)
pass the naive global lat/lng validity check but distort per-ZIP aggregation. Filtered
here before aggregating, exactly as validated in EDA cell 120–121.

In [ ]:
BRAZIL_LAT_MIN, BRAZIL_LAT_MAX = -34, 6
BRAZIL_LNG_MIN, BRAZIL_LNG_MAX = -74, -33

geo_valid = geolocation_cleaned[
    geolocation_cleaned['geolocation_lat'].between(BRAZIL_LAT_MIN, BRAZIL_LAT_MAX) &
    geolocation_cleaned['geolocation_lng'].between(BRAZIL_LNG_MIN, BRAZIL_LNG_MAX)
].copy()

print(f"geolocation rows before Brazil-bounds filter: {len(geolocation_cleaned)}")
print(f"geolocation rows after filter: {len(geo_valid)} (removed: {len(geolocation_cleaned) - len(geo_valid)})")

# Median is used (not mean)
geo_avg = geo_valid.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'median'),
    lng=('geolocation_lng', 'median')
).reset_index()

print(f"unique ZIP prefixes with valid coordinates: {len(geo_avg)}")

geolocation rows before Brazil-bounds filter: 1000163
geolocation rows after filter: 1000121 (removed: 42)
unique ZIP prefixes with valid coordinates: 19010


## 5. Haversine Distance (Buyer ↔ Seller)

In [8]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

sellers_geo = sellers_cleaned.merge(
    geo_avg, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left'
).rename(columns={'lat': 'seller_lat', 'lng': 'seller_lng'})

customer_geo = customers_cleaned.merge(
    geo_avg, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left'
).rename(columns={'lat': 'customer_lat', 'lng': 'customer_lng'})

item_distance = order_items_cleaned.merge(
    sellers_geo[['seller_id', 'seller_lat', 'seller_lng']], on='seller_id', how='left'
).merge(
    orders_cleaned[['order_id', 'customer_id']], on='order_id', how='left'
).merge(
    customer_geo[['customer_id', 'customer_lat', 'customer_lng']], on='customer_id', how='left'
)

item_distance['distance_km'] = haversine_distance(
    item_distance['seller_lat'], item_distance['seller_lng'],
    item_distance['customer_lat'], item_distance['customer_lng']
)

print(f"item_distance rows: {len(item_distance)}  (should equal order_items_cleaned: {len(order_items_cleaned)})")
print(f"rows with valid distance_km: {item_distance['distance_km'].notna().sum()}")

item_distance rows: 112650  (should equal order_items_cleaned: 112650)
rows with valid distance_km: 112095


## 6. `items_master` — Item-level Analytical Table

One row per `order_item`. Combines product attributes (category, weight, dimensions,
volume), the buyer-seller distance, and order-level context (`is_late`, `review_score`)
attached — not aggregated — so each item retains its own product-level detail.

**Important:** `is_late` and `review_score` are order-level facts repeated across every
item of a multi-item order. This is intentional (needed for product/category ↔
satisfaction analysis) but must be documented — do not sum or average `review_score`
across items of the same order, as that would double count.

In [ ]:
items_master = order_items_cleaned.merge(
    products_cleaned[[
        'product_id', 'product_category_name_english',
        'product_weight_g', 'product_length_cm',
        'product_height_cm', 'product_width_cm',
        'product_photos_qty', 'product_description_lenght'
    ]],
    on='product_id', how='left'
)

# Volume feature (validated in EDA cell 116)
items_master['volume_cm3'] = (
    items_master['product_length_cm'] * items_master['product_height_cm'] * items_master['product_width_cm']
)

# Freight-to-price ratio (validated in EDA cell 35)
items_master['freight_ratio'] = items_master['freight_value'] / items_master['price']

# Distance feature (validated in EDA cells 120-127)
items_master = items_master.merge(
    item_distance[['order_id', 'order_item_id', 'distance_km']],
    on=['order_id', 'order_item_id'], how='left'
)

# Order-level context attached (not aggregated) for product/category-vs-satisfaction analysis
items_master = items_master.merge(orders_fe[['order_id', 'is_late']], on='order_id', how='left')
items_master = items_master.merge(reviews_fe, on='order_id', how='left')

print(f"items_master shape: {items_master.shape}")
print(f"should equal order_items_cleaned row count: {len(order_items_cleaned)}")
assert len(items_master) == len(order_items_cleaned), "Row count mismatch — a merge duplicated rows!"
print("Row-count check passed: no duplication introduced by merges.")

items_master shape: (112650, 19)
should equal order_items_cleaned row count: 112650
Row-count check passed: no duplication introduced by merges.


In [10]:
items_master.sample(5)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty,product_description_lenght,volume_cm3,freight_ratio,distance_km,is_late,review_score
28990,41f25580214be3f00eb80620ca971032,2,c94d19d021d4a8fbae5a45f904aeea90,701938c450705b8ae65fc923b70f35c7,2017-02-04 21:57:33,103.99,37.24,furniture_living_room,9750.0,27.0,15.0,92.0,4.0,612.0,37260.0,0.358111,681.339247,False,5.0
31141,46a354ecffd29742e985db9b49cae302,1,e5215437680ffc2cfdb0ce9088870176,cc3d14116804f7209f3884f6367f1152,2018-02-09 12:55:42,29.90,14.10,fashion_shoes,500.0,30.0,5.0,12.0,2.0,1757.0,1800.0,0.471572,358.130279,False,5.0
106936,f307eeea250efc6c6bdd863ae32162b5,1,a75b732b129c2fe3d8e371dd8333cd00,78a4d36f65a019172c6154468831a69f,2017-11-30 11:15:23,209.90,21.44,kitchen_dining_laundry_garden_furniture,6400.0,16.0,2.0,11.0,5.0,731.0,352.0,0.102144,270.779529,False,5.0
39304,5958ed39b1c52e9a1f4a4cbfcfcc6506,1,c21bd5b6be0663cddb80ec6744d6ca24,e9779976487b77c6d4ac45f75ec7afe9,2017-11-17 17:07:01,99.49,8.44,sports_leisure,538.0,23.0,17.0,17.0,3.0,3315.0,6647.0,0.084833,40.324355,False,5.0
89803,cbff3990ac7d2b2026c2e3a639aaabb9,1,8c591ab0ca519558779df02023177f44,a1043bafd471dff536d0c462352beb48,2018-02-21 12:47:45,120.00,26.46,garden_tools,6050.0,20.0,20.0,20.0,1.0,1893.0,8000.0,0.220500,467.763095,False,4.0


## 7. `orders_master` — Order-level Analytical Table

One row per `order_id`. Item-level facts (price, freight, weight, distance) are
aggregated to order level first (`sum` for additive quantities, `mean` for distance),
then joined to the order lifecycle/delivery features and the deduplicated review score.

For the ~90% of orders with a single item, these aggregates equal the item-level value
exactly — no information is lost for the vast majority of the dataset.

In [11]:
order_agg = items_master.groupby('order_id').agg(
    n_items=('order_item_id', 'count'),
    total_item_price=('price', 'sum'),
    total_freight_value=('freight_value', 'sum'),
    avg_distance_km=('distance_km', 'mean'),
    total_weight_g=('product_weight_g', 'sum')
).reset_index()

orders_master = (
    orders_fe
    .merge(order_agg, on='order_id', how='left')
    .merge(reviews_fe, on='order_id', how='left')
)

print(f"orders_master shape: {orders_master.shape}")
print(f"should equal orders_cleaned row count: {len(orders_cleaned)}")
assert len(orders_master) == len(orders_cleaned), "Row count mismatch — a merge duplicated rows!"
print("Row-count check passed: no duplication introduced by merges.")

orders_master shape: (99441, 23)
should equal orders_cleaned row count: 99441
Row-count check passed: no duplication introduced by merges.


## 8. Final Validation

Before exporting, run the same checks applied throughout this project: shape,
missing values on the newly engineered columns, and a sanity check against numbers
already validated in EDA.

In [13]:
print("=== items_master ===")
print(items_master.shape)
print(items_master[['product_weight_g','distance_km','review_score','is_late']].isna().sum())

print("\n=== orders_master ===")
print(orders_master.shape)
print(orders_master[['total_freight_value','avg_distance_km','review_score','is_late']].isna().sum())

# Spot-check against EDA cell 102 result (mean review score by lateness)
check = orders_master.dropna(subset=['review_score', 'is_late']).groupby('is_late')['review_score'].mean()
print("\nSpot-check vs EDA cell 102 (expected ~4.29 / ~2.27):")
print(check)

=== items_master ===
(112650, 19)
product_weight_g     18
distance_km         555
review_score        942
is_late               0
dtype: int64

=== orders_master ===
(99441, 23)
total_freight_value     775
avg_distance_km        1265
review_score            768
is_late                   0
dtype: int64

Spot-check vs EDA cell 102 (expected ~4.29 / ~2.27):
is_late
False    4.211797
True     2.270918
Name: review_score, dtype: float64


## 9. Export

In [14]:
items_master.to_csv('../data/processed/items_master.csv', index=False)
orders_master.to_csv('../data/processed/orders_master.csv', index=False)
print("Saved items_master.csv and orders_master.csv to ../data/processed/")

Saved items_master.csv and orders_master.csv to ../data/processed/
